# Junções, integração e reprodutibilidade

## 1. Chaves e cardinalidade

Antes de juntar, declare a unidade de cada tabela e a cardinalidade esperada:
1:1, 1:N ou N:N. Uma chave duplicada pode multiplicar linhas e fabricar peso
analítico. `validate` testa a hipótese de cardinalidade; `indicator` mostra a
cobertura da correspondência.

In [ ]:
import json
import pandas as pd

catalogo = pd.read_csv("dados/intermediarios/catalogo_normalizado.csv", dtype={"codigo_municipio": "string"})
municipios = pd.read_csv("dados/brutos/extrato_codigos_municipios_ibge.csv", dtype={"codigo_municipio": "string"})
integrada = catalogo.merge(
    municipios, on="codigo_municipio", how="left",
    validate="many_to_one", indicator=True,
)
integrada["_merge"].value_counts()

### Auditoria da junção

`left_only` não deve ser descartado automaticamente: pode indicar código
inválido, cobertura incompleta da tabela de referência ou mudança temporal.
Compare contagens antes e depois e examine chaves sem correspondência.

In [ ]:
auditoria_juncao = {
    "linhas_antes": len(catalogo),
    "linhas_depois": len(integrada),
    "sem_correspondencia": int(integrada["_merge"].eq("left_only").sum()),
    "ids_unicos_antes": int(catalogo["id_documento"].nunique()),
    "ids_unicos_depois": int(integrada["id_documento"].nunique()),
}
pd.Series(auditoria_juncao)

## 2. Integrar textos, metadados e indicadores

Um documento pode possuir zero ou um arquivo textual nesta versão, vários
temas e vários indicadores. Vamos criar uma tabela textual 1:1 apenas para os
arquivos disponíveis; temas e indicadores permanecem em tabelas longas para
evitar colunas multivaloradas.

In [ ]:
from pathlib import Path

metadados = json.loads(Path("dados/brutos/metadados.json").read_text(encoding="utf-8"))
textos = []
temas = []
for item in metadados:
    for tema in item["temas"]:
        temas.append({"id_documento": item["id_documento"], "tema": tema})
    if item["arquivo_texto"]:
        caminho = Path("dados/brutos") / item["arquivo_texto"]
        textos.append({"id_documento": item["id_documento"], "texto": caminho.read_text(encoding="utf-8")})
tabela_textos = pd.DataFrame(textos)
tabela_temas = pd.DataFrame(temas)
base_documentos = integrada.merge(tabela_textos, on="id_documento", how="left", validate="one_to_one")
print("Documentos:", len(base_documentos), "| relações documento-tema:", len(tabela_temas))

In [ ]:
largo = pd.read_csv("dados/brutos/indicadores_largos.csv")
indicadores = largo.melt(id_vars="id_documento", var_name="tema_periodo", value_name="ocorrencias")
partes = indicadores["tema_periodo"].str.extract(r"(?P<tema>.+)_(?P<periodo>\d{4})")
indicadores = pd.concat([indicadores[["id_documento", "ocorrencias"]], partes], axis=1)

base_documentos.drop(columns="_merge").to_csv("dados/derivados/documentos_processaveis.csv", index=False)
tabela_temas.to_csv("dados/derivados/documentos_temas.csv", index=False)
indicadores.to_csv("dados/derivados/indicadores_longos.csv", index=False)

## 3. Organização e execução

Uma estrutura simples separa entrada, intermediários e derivados. Notebooks
numerados tornam a ordem visível, mas reprodutibilidade também requer ambiente,
parâmetros, versões e execução desde o início. Saídas derivadas devem poder ser
reconstruídas sem editar manualmente células intermediárias.

In [ ]:
from pathlib import Path

verificacoes = {
    "ids_documentos_unicos": base_documentos["id_documento"].is_unique,
    "nenhuma_linha_criada_na_juncao_municipal": len(base_documentos) == len(catalogo),
    "temas_referenciam_documentos": set(tabela_temas["id_documento"]).issubset(set(base_documentos["id_documento"])),
    "arquivos_derivados": len(list(Path("dados/derivados").glob("*.csv"))),
}
verificacoes

## Atividade — plano de integração

Desenhe as tabelas, unidades, chaves, cardinalidades, campos compartilhados,
validações, tratamento de não correspondências e saídas. Indique como um
resultado será rastreado até a fonte. **Plano:** Escreva aqui.

## Síntese

Junção é uma afirmação de identidade e relação, não mero encaixe de colunas.
A base processável pode ser plural: tabela de documentos, tabela de relações,
textos e indicadores ligados por chaves verificadas.